In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name, monotonically_increasing_id, to_date
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DateType, BooleanType
import duckdb as db

In [19]:
con = db.connect("../../ProjectData.duckdb")

con.execute("""
    CREATE TABLE IF NOT EXISTS silver.cleaned_data (
        product_id VARCHAR,
        product_parent BIGINT,
        product_title VARCHAR,
        vine VARCHAR,
        verified_purchase VARCHAR,
        review_headline VARCHAR,
        review_body VARCHAR,
        review_date DATE,
        marketplace_id BIGINT,
        product_category_id BIGINT,
        label BOOLEAN,
        _ingested_at TIMESTAMP,
        _source_file VARCHAR,
        _index BIGINT
    )
""")

# con.close()

In [21]:
rows = con.execute("""
    SELECT * FROM bronze.reviews_raw
""").df()
rows

,product_id,product_parent,product_title,vine,verified_purchase,review_headline,review_body,review_date,marketplace_id,product_category_id,label,_corrupt_record,_ingested_at,_load_date,_source_file,_index
0,B0001NDG70,796529195,the tarantino connection,N,N,True Romance - Coffret Collector 3 DVD [Éditio...,D'accord Tony scott est un réalisateur ringard...,2004-05-15,2,3,True,None,2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,742
1,B004UID2LC,479653847,FREEdi YouTube Downloader,N,Y,Super App!!,Dér Dớwnlớád dér YớúTúbé-Vidéớs áls mp4-Vidéớ ...,2013-09-29,0,1,False,None,2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,130
2,B004XAU7VA,678332854,Thor [Combo Blu-ray + DVD],N,N,None,Je complète ma collection avec celle de Captai...,2013-05-26,2,3,False,None,2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,1151
3,B0002EQU6C,223858674,Artic Silver 5 Wärmeleitpaste - 3.5 g,N,Y,Géniálé Pásté!,Habe diese Paste schon lange im Einsatz und bi...,2015-01-10,0,5,False,None,2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,564
4,B00005AXK7,406550936,Hercules (Special Collection),N,Y,disney klassiker :),"Nicht wie aus der mythologie, aber sehr schön....",2014-07-01,0,3,False,None,2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9609,B000GBEYWS,496520461,La Petite Sirène,N,Y,Excellent produit,"C'est un excellent produit. Je suis content, p...",2011-09-07,2,3,False,None,2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,60129542713
9610,B0084I9QIW,897848227,Michael Jackson : Live at Wembley July 16 1988,N,Y,sublime concert BAD de MJ THE BEST,TRES STISFAITE DE CE DVD ..ETANT FAN DE MJ..j...,2014-05-11,2,3,True,None,2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,60129542615
9611,B00004THG5,724086367,Deep Blue Sea,N,N,Billiger Computerkitsch im Actionfahrwasser,"""Dieser Film ist in vielerlei Beziehung ein Fe...",NaT,<NA>,<NA>,<NA>,"1555,B00004THG5,724086367,Deep Blue Sea,N,N,Bi...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,60129542305
9612,B00I6IKSZ0,918624977,Pixel Gun 3D (Pocket Edition) - multiplayer sh...,N,Y,"Addicting,overwhelming,Awesome",I got this game when it first came out and wat...,2014-12-12,1,1,False,None,2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,60129542225


In [17]:
con.execute("SELECT * FROM bronze.reviews_raw WHERE _corrupt_record IS NOT NULL").df()

,product_id,product_parent,product_title,vine,verified_purchase,review_headline,review_body,review_date,marketplace_id,product_category_id,label,_corrupt_record,_ingested_at,_load_date,_source_file,_index
0,B002VTOAPO,946348526,Damages - Saison 2 - Coffret 3 DVD,N,Y,None,"""J'ai d&eacute;couvert cette s&eacute;rie par ...",NaT,<NA>,<NA>,<NA>,"3618,B002VTOAPO,946348526,Damages - Saison 2 -...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,369
1,B0050ID9E6,678332854,Thor (+ DVD) [Blu-ray],N,N,Bofrost hat mehr Geschmack,"""Dié Rézénsiớn béziéht sich áúf dié Blúráy Vér...",NaT,<NA>,<NA>,<NA>,"9445,B0050ID9E6,678332854,Thor (+ DVD) [Blu-ra...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,968
2,B000935TV8,233458585,Une Symphonie Imaginaire/Eine Imaginäre Sinfonie,N,N,Gut fingiert!,"""Immer wieder eine Freude: Die Musiciens du Lo...",NaT,<NA>,<NA>,<NA>,"3365,B000935TV8,233458585,Une Symphonie Imagin...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,343
3,B0045HB5L2,553080263,Battlestar Galactica Komplettbox (25 Disc) [Li...,N,Y,Gigantisch,"""Ich spare mir das hin- und her über die Plast...",NaT,<NA>,<NA>,<NA>,"9515,B0045HB5L2,553080263,Battlestar Galactica...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,973
4,0307463745,877463784,Selbstverliebt und ohne Tiefgang,N,N,Rework,"""Auf den ersten Blick erschien mir das Buch al...",NaT,<NA>,<NA>,<NA>,"4398,0307463745,877463784,Selbstverliebt und o...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1046,B00008Z5GE,661969053,Tunnel of Love,N,N,Springsteens persönlichstes Werk,"""Nach dem Mega-Seller \""""Born In The USA\\""""",NaT,<NA>,<NA>,<NA>,"10549,B00008Z5GE,661969053,Tunnel of Love,N,N,...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,60129543145
1047,B002JHVPDI,584084882,"Ungew&ouml;hnliche Serie! Spannend, fesselnd, ...",N,Y,Dexter - Die erste Season [4 DVDs],"""Mir wurde von Freunden diese Serie \""""Dexter\...",NaT,<NA>,<NA>,<NA>,"6622,B002JHVPDI,584084882,""Ungew&ouml;hnliche ...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,60129542785
1048,B00D93NFDE,675313834,Blackfield IV -Blackfield,N,N,complainte douce-amère,"""On sent dès les premières notes, que l'ami Wi...",NaT,<NA>,<NA>,<NA>,"10847,B00D93NFDE,675313834,Blackfield IV -Blac...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,60129543172
1049,B00064XC6E,294528259,None,N,N,Le Bal des vampires,"""Ce film malgré son âge ne perd pas une ride<b...",NaT,<NA>,2,<NA>,"4241,B00064XC6E,294528259,,N,N,Le Bal des vamp...",2026-02-28 18:22:23.138346+01:00,2026-02-28,file:///home/ronlakeman/MasterInformationStudi...,60129542559


In [23]:
con.close()

In [22]:
# Analyze corrupt records to understand failure patterns
print("=" * 80)
print("CORRUPT RECORDS ANALYSIS")
print("=" * 80)

# Get first 20 corrupt records
print("\n1. Sample of corrupt records (first 20):")
print("-" * 80)
corrupt_samples = con.execute("""
    SELECT 
        _corrupt_record as raw_record
    FROM bronze.reviews_raw
    WHERE _corrupt_record IS NOT NULL
    LIMIT 20
""").fetchall()

for i, (record,) in enumerate(corrupt_samples, 1):
    # Truncate long records for readability
    display = record[:150] + "..." if len(record) > 150 else record
    print(f"{i:2d}. {display}")

# Analyze patterns
print("\n2. Corrupt record characteristics:")
print("-" * 80)
stats = con.execute("""
    SELECT 
        COUNT(*) as total_corrupt,
        MIN(LENGTH(_corrupt_record)) as min_length,
        MAX(LENGTH(_corrupt_record)) as max_length,
        AVG(LENGTH(_corrupt_record))::INT as avg_length,
        SUM(CASE WHEN _corrupt_record LIKE '%""%' THEN 1 ELSE 0 END) as has_quotes,
        SUM(CASE WHEN _corrupt_record LIKE '%,%' THEN 1 ELSE 0 END) as has_commas,
        SUM(CASE WHEN LENGTH(_corrupt_record) < 50 THEN 1 ELSE 0 END) as very_short_records
    FROM bronze.reviews_raw
    WHERE _corrupt_record IS NOT NULL
""").fetchone()

print(f"Total corrupt records: {stats[0]}")
print(f"Record length range: {stats[1]} - {stats[2]} chars")
print(f"Average record length: {stats[3]} chars")
print(f"Records with quotes: {stats[4]}")
print(f"Records with commas: {stats[5]}")
print(f"Very short records (<50 chars): {stats[6]}")

# Find shortest corrupt records (often indicates parsing failure)
print("\n3. Shortest corrupt records (likely parsing failures):")
print("-" * 80)
short_records = con.execute("""
    SELECT 
        LENGTH(_corrupt_record) as length,
        _corrupt_record as record
    FROM bronze.reviews_raw
    WHERE _corrupt_record IS NOT NULL
    ORDER BY LENGTH(_corrupt_record) ASC
    LIMIT 10
""").fetchall()

for length, record in short_records:
    print(f"[{length:3d} chars] {record}")

# Check for common issues
print("\n4. Potential issue categories:")
print("-" * 80)
issues = con.execute("""
    SELECT 
        CASE 
            WHEN LENGTH(_corrupt_record) < 20 THEN 'Very short/incomplete'
            WHEN _corrupt_record LIKE '%"%' THEN 'Quote parsing issues'
            WHEN LENGTH(_corrupt_record) > 5000 THEN 'Extremely long field'
            ELSE 'Field count mismatch'
        END as issue_type,
        COUNT(*) as count
    FROM bronze.reviews_raw
    WHERE _corrupt_record IS NOT NULL
    GROUP BY issue_type
    ORDER BY count DESC
""").fetchall()

for issue, count in issues:
    print(f"  • {issue}: {count} records")

con.close()

CORRUPT RECORDS ANALYSIS

1. Sample of corrupt records (first 20):
--------------------------------------------------------------------------------
 1. 3618,B002VTOAPO,946348526,Damages - Saison 2 - Coffret 3 DVD,N,Y,,"J'ai d&eacute;couvert cette s&eacute;rie par hasard et j'ai accroch&eacute;. A chaq...
 2. 9445,B0050ID9E6,678332854,Thor (+ DVD) [Blu-ray],N,N,Bofrost hat mehr Geschmack,"Dié Rézénsiớn béziéht sich áúf dié Blúráy Vérsiớn, dié ich mir áúslié...
 3. 3365,B000935TV8,233458585,Une Symphonie Imaginaire/Eine Imaginäre Sinfonie,N,N,Gut fingiert!,"Immer wieder eine Freude: Die Musiciens du Louvre spiele...
 4. 9515,B0045HB5L2,553080263,Battlestar Galactica Komplettbox (25 Disc) [Limited Edition] [25 DVDs],N,Y,Gigantisch,"Ich spare mir das hin- und her über d...
 5. 4398,0307463745,877463784,Selbstverliebt und ohne Tiefgang,N,N,Rework,"Auf den ersten Blick erschien mir das Buch als nette Zusammenfassung guter Rats...
 6. 5277,B003IYWRT2,505469654,Ich kann es kaum glauben...,N,N,